# `code_storage.ipynb` — provenance for committed inputs (not part of the main run)

This notebook is not part of the reproducible pipeline. `scripts/data_merge.ipynb` is the
pipeline. This file collects the one-off scripts that produced several of the committed input
files under `data/`, kept here so the origin of every input is documented.

**Inputs this notebook produced (already committed under `data/`):**

| Section | Produces | From |
|---|---|---|
| Physical metric from Microsoft's data | `tasks_dwa_iwa_gwa_v30.1_physical.csv` | `physical_tasks.csv` + O\*NET taxonomy, with DWA majority-rule imputation for unmapped tasks |
| Collaboration patterns from AEI v3/v4/v5 | `automation_vs_augmentation_by_task_{v3,v4,v5,api_v3,api_v4,api_v5}.csv` | the raw `aei_raw_*` files (pivots the five collaboration-pattern percentages per task) |
| ECO Pct CSVs | `task_pct_eco_{2015,2025}.csv` | O\*NET `task_statements_v{20.1,30.1}.csv` (unique normalized tasks, `pct` initialized to 0 — the economy baseline) |
| Master Pct Normalized | `master_pct_normalized.csv` | per-title mean AI-usage share across the AEI/Microsoft `first_pass_*` files (consumed by `adjust_emp` for decimal-SOC employment reallocation) |

The remaining cells (GWA pct for debiasing — Copilot / ChatGPT / Claude distributions) are the
calculations supporting the usage-debiasing analysis in the paper, not input generators.

**Notes for re-running these cells.** They are kept for provenance, not as turnkey scripts. The
physical-flags cell runs from committed inputs (`physical_tasks.csv`, `task_statements_v29.0.csv`,
`tasks_dwa_iwa_gwa_v30.1.csv`). Two cells depend on files generated by the main pipeline and will
not run on a fresh clone until it has been run:
- the Master Pct cell reads `first_pass_aei_*` / `first_pass_microsoft`, produced by Part 1 of
  `data_merge.ipynb`. `master_pct_normalized.csv` is committed as a fixed artifact that breaks a
  circular dependency: Part 1's `adjust_emp` needs it, but it was itself built from Part 1's
  outputs. Use the committed copy; do not regenerate it mid-run.
- the Claude GWA cell reads a `data/final/final_aei_all_usage_*.csv` output that exists only after
  a full pipeline run.

## Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textwrap import wrap
import numpy as np
import re
import os
from pathlib import Path

#Helper Functions

def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()                   # lowercase + trim
    text = re.sub(r"[^\w\s]", "", text)            # remove punctuation
    text = re.sub(r"\s+", " ", text)               # collapse multiple spaces
    return text


## Physical metric from Microsoft's data to tasks_dwa_iwa_gwa

In [ ]:
import pandas as pd
import numpy as np

# Load the three files
physical_tasks = pd.read_csv("../data/physical_tasks.csv")
task_statements = pd.read_csv("../data/task_statements_v29.0.csv")
tasks_dwa = pd.read_csv("../data/tasks_dwa_iwa_gwa_v30.1.csv")

# --- STEP 1: Merge Physical into Task Statements ---
merged_statements = pd.merge(
    task_statements, 
    physical_tasks[['Task ID', 'Physical']], 
    on='Task ID', 
    how='left'
)

# --- STEP 2: Create mapping using both Title (Occupation) and Task ---
# We pull 'Title', 'Task', and 'Physical' to ensure the merge is specific to the occupation context
mapping_subset = merged_statements[['Title', 'Task', 'Physical']].drop_duplicates()

# --- STEP 3: Merge on Title and Task simultaneously ---
# Assuming tasks_dwa has 'title' for occupation and 'task' for the task description
df = pd.merge(
    tasks_dwa,
    mapping_subset,
    left_on=['Title', 'task'],
    right_on=['Title', 'Task'],
    how='left'
).drop(columns=['Task'])

# Impute missing values based on DWA majority (mode)
dwa_majority = df.groupby('dwa_title')['Physical'].agg(
    lambda x: x.mode()[0] if not x.mode().empty else np.nan
)

# Apply imputation and cleanup columns
df.loc[df['Physical'].isna(), 'Physical'] = df.loc[df['Physical'].isna(), 'dwa_title'].map(dwa_majority)
df = df.rename(columns={'Physical': 'physical'})

# Save result
df.to_csv("../data/tasks_dwa_iwa_gwa_v30.1_physical.csv", index=False)

df

,task_id,task,dwa_id,dwa_title,iwa_id,iwa_title,gwa_id,gwa_title,O*NET-SOC Code,Title,Date,Domain Source,physical
0,20461,"Review and analyze legislation, laws, or publi...",4.A.2.a.4.I09.D03,Analyze impact of legal or regulatory changes.,4.A.2.a.4.I09,Assess characteristics or impacts of regulatio...,4.A.2.a.4,Analyzing Data or Information,11-1011.00,Chief Executives,07/2014,Analyst,False
1,20461,"Review and analyze legislation, laws, or publi...",4.A.4.b.6.I08.D04,Advise others on legal or regulatory complianc...,4.A.4.b.6.I08,Advise others on legal or regulatory matters.,4.A.4.b.6,Providing Consultation and Advice to Others,11-1011.00,Chief Executives,07/2014,Analyst,False
2,8823,Direct or coordinate an organization's financi...,4.A.4.b.4.I09.D02,Direct financial operations.,4.A.4.b.4.I09,Manage budgets or finances.,4.A.4.b.4,"Guiding, Directing, and Motivating Subordinates",11-1011.00,Chief Executives,03/2014,Analyst,False
3,8824,"Confer with board members, organization offici...",4.A.4.a.2.I03.D14,Confer with organizational members to accompli...,4.A.4.a.2.I03,Communicate with others about operational plan...,4.A.4.a.2,"Communicating with Supervisors, Peers, or Subo...",11-1011.00,Chief Executives,03/2014,Analyst,False
4,8825,Analyze operations to evaluate performance of ...,4.A.2.a.4.I07.D09,Analyze data to assess operational or project ...,4.A.2.a.4.I07,Analyze data to improve operations.,4.A.2.a.4,Analyzing Data or Information,11-1011.00,Chief Executives,03/2014,Analyst,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23846,12807,Unload cars containing liquids by connecting h...,4.A.3.a.2.I34.D01,Connect hoses to equipment or machinery.,4.A.3.a.2.I34,Connect components or supply lines to equipmen...,4.A.3.a.2,Handling and Moving Objects,53-7121.00,"Tank Car, Truck, and Ship Loaders",03/2014,Analyst,True
23847,12808,Copy and attach load specifications to loaded ...,4.A.1.b.1.I01.D03,Mark materials or objects for identification.,4.A.1.b.1.I01,Mark materials or objects for identification.,4.A.1.b.1,"Identifying Objects, Actions, and Events",53-7121.00,"Tank Car, Truck, and Ship Loaders",03/2014,Analyst,True
23848,12809,Start pumps and adjust valves or cables to reg...,4.A.3.a.3.I02.D03,Control pumps or pumping equipment.,4.A.3.a.3.I02,Operate pumping systems or equipment.,4.A.3.a.3,Controlling Machines and Processes,53-7121.00,"Tank Car, Truck, and Ship Loaders",03/2014,Analyst,True
23849,12810,"Perform general warehouse activities, such as ...",4.A.1.b.3.I01.D14,Weigh materials to ensure compliance with spec...,4.A.1.b.3.I01,"Measure physical characteristics of materials,...",4.A.1.b.3,Estimating the Quantifiable Characteristics of...,53-7121.00,"Tank Car, Truck, and Ship Loaders",03/2014,Analyst,True


## Getting collaboration patterns from v3 and v4 and v5 aei data

In [3]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data")

files = {
    "aei_raw_1p_api_2025-08-04_to_2025-08-11.csv": "automation_vs_augmentation_by_task_api_v3.csv",
    "aei_raw_1p_api_2025-11-13_to_2025-11-20.csv": "automation_vs_augmentation_by_task_api_v4.csv",
    "aei_raw_1p_api_2026-02-05_to_2026-02-12.csv": "automation_vs_augmentation_by_task_api_v5.csv",
    "aei_raw_claude_ai_2025-08-04_to_2025-08-11.csv": "automation_vs_augmentation_by_task_v3.csv",
    "aei_raw_claude_ai_2025-11-13_to_2025-11-20.csv": "automation_vs_augmentation_by_task_v4.csv",
    "aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv": "automation_vs_augmentation_by_task_v5.csv",
}

for f, out_name in files.items():

    df = pd.read_csv(data_dir / f)

    df = df[
        (df["geo_id"] == "GLOBAL") &
        (df["facet"] == "onet_task::collaboration") &
        (df["variable"] == "onet_task_collaboration_pct")
    ].copy()

    split = df["cluster_name"].str.rsplit("::", n=1, expand=True)
    df["task"] = split[0]
    df["collab_type"] = split[1]

    out = (
        df
        .pivot_table(
            index="task",
            columns="collab_type",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )
    out.iloc[:, 1:] = out.iloc[:, 1:] / 100

    out.columns.name = None
    out.columns = out.columns.str.replace(" ", "_")

    out.to_csv(data_dir / out_name, index=False)

    print(f"Saved: {out_name}")

Saved: automation_vs_augmentation_by_task_api_v3.csv
Saved: automation_vs_augmentation_by_task_api_v4.csv
Saved: automation_vs_augmentation_by_task_api_v5.csv
Saved: automation_vs_augmentation_by_task_v3.csv
Saved: automation_vs_augmentation_by_task_v4.csv
Saved: automation_vs_augmentation_by_task_v5.csv


In [ ]:
# import pandas as pd
# from pathlib import Path
# import numpy as np

# data_dir = Path("../data")

# files = [
#     "aei_raw_1p_api_2025-08-04_to_2025-08-11_collaboration_tasks.csv",
#     "aei_raw_1p_api_2025-11-13_to_2025-11-20_collaboration_tasks.csv",
#     "aei_raw_claude_ai_2025-08-04_to_2025-08-11_collaboration_tasks.csv",
#     "aei_raw_claude_ai_2025-11-13_to_2025-11-20_collaboration_tasks.csv",
# ]

# cols = [
#     "directive",
#     "feedback loop",
#     "learning",
#     "none",
#     "not_classified",
#     "task iteration",
#     "validation",
# ]

# for f in files:
#     df = pd.read_csv(data_dir / f)

#     df["row_sum"] = df[cols].sum(axis=1)

#     bad = df[~np.isclose(df["row_sum"], 100, atol=1e-6)]

#     print(f"\nFile: {f}")
#     print(f"Total rows: {len(df)}")
#     print(f"Rows NOT summing to 100: {len(bad)}")

#     if len(bad) > 0:
#         print(bad[["task", "row_sum"]].head())


File: aei_raw_1p_api_2025-08-04_to_2025-08-11_collaboration_tasks.csv
Total rows: 2055
Rows NOT summing to 100: 0

File: aei_raw_1p_api_2025-11-13_to_2025-11-20_collaboration_tasks.csv
Total rows: 2252
Rows NOT summing to 100: 0

File: aei_raw_claude_ai_2025-08-04_to_2025-08-11_collaboration_tasks.csv
Total rows: 2617
Rows NOT summing to 100: 0

File: aei_raw_claude_ai_2025-11-13_to_2025-11-20_collaboration_tasks.csv
Total rows: 3169
Rows NOT summing to 100: 0


## Eco Pct Csvs

In [7]:

DATA_DIR = Path("../data")

# Configuration for the two files
files_to_process = [
    ("task_statements_v20.1.csv", "task_pct_eco_2015.csv"),
    ("task_statements_v30.1.csv", "task_pct_eco_2025.csv")
]

for input_file, output_file in files_to_process:
    file_path = DATA_DIR / input_file
    
    if file_path.exists():
        # 1. Load only the 'Task' column
        df = pd.read_csv(file_path, usecols=["Task"])
        
        # 2. Apply normalization and get unique values
        unique_tasks = df["Task"].apply(normalize_text).unique()
        
        # 3. Create final DataFrame with renamed column and new pct column
        final_df = pd.DataFrame({
            "task_name": unique_tasks,
            "pct": 0  # Initializes all rows with 0
        })
        
        # 4. Save
        final_df.to_csv(DATA_DIR / output_file, index=False)
        print(f"Processed {input_file} -> Saved {len(final_df)} unique tasks to {output_file}")
    else:
        print(f"Warning: {input_file} not found in {DATA_DIR}")

print("\nTask extraction with pct column complete.")

Processed task_statements_v20.1.csv -> Saved 18405 unique tasks to task_pct_eco_2015.csv
Processed task_statements_v30.1.csv -> Saved 17507 unique tasks to task_pct_eco_2025.csv

Task extraction with pct column complete.


## Master Pct Normalized for Standard Detailed Occ Emp Number Adjustment

In [13]:
DATA_DIR = Path("../data")

# 1. Identify the target files (Excluding MCP and ECO)
# Based on your image: AEI (v1, v2, v3, v4, api_v3, api_v4) and Microsoft
target_prefixes = ["first_pass_aei", "first_pass_microsoft"]
files = [f for f in os.listdir(DATA_DIR) if f.endswith(".csv") and 
         any(f.startswith(pre) for pre in target_prefixes) and 
         not any(x in f for x in ["mcp", "eco", "api"])]

all_sums = []

# 2. Process each file individually
for file in files:
    df = pd.read_csv(DATA_DIR / file)
    
    # Check if required columns exist
    if 'title' in df.columns and 'pct_normalized' in df.columns:
        # Sum pct_normalized by title for THIS specific file
        file_sum = df.groupby('title')['pct_normalized'].sum().reset_index()
        file_sum['source_file'] = file
        all_sums.append(file_sum)
        print(f"Processed {file}: {len(file_sum)} unique titles found.")

# 3. Combine all summed data
combined_sums = pd.concat(all_sums, ignore_index=True)

# 4. Create the final Master DF
# We group by 'title' and average the sums collected from the different files
master_title_df = combined_sums.groupby('title')['pct_normalized'].mean().reset_index()

# Rename for clarity
master_title_df.columns = ['title', 'avg_total_pct_normalized']
master_title_df['avg_total_pct_normalized'] = master_title_df['avg_total_pct_normalized'] / master_title_df['avg_total_pct_normalized'].sum() 

# 5. Save the result
output_path = DATA_DIR / "master_pct_normalized.csv"
master_title_df.to_csv(output_path, index=False)

print(f"\nSuccess! Master list created with {len(master_title_df)} unique titles.")
print(f"Saved to: {output_path}")

# Display top examples
print("\nTop 5 Titles by AI Intensity:")
print(master_title_df.sort_values('avg_total_pct_normalized', ascending=False).head())

Processed first_pass_aei_v1.csv: 756 unique titles found.
Processed first_pass_aei_v2.csv: 749 unique titles found.
Processed first_pass_aei_v3.csv: 699 unique titles found.
Processed first_pass_aei_v4.csv: 722 unique titles found.
Processed first_pass_microsoft.csv: 890 unique titles found.

Success! Master list created with 1007 unique titles.
Saved to: ..\data\master_pct_normalized.csv

Top 5 Titles by AI Intensity:
                                     title  avg_total_pct_normalized
885  Software Developers, Systems Software                  0.097032
884      Software Developers, Applications                  0.083247
992                         Web Developers                  0.059143
177                   Computer Programmers                  0.048246
236           Data Warehousing Specialists                  0.024561


In [10]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("../data")

def normalize_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

# 1. Load the data
# Usecols saves memory and prevents errors if other columns are messy
master_df = pd.read_csv(DATA_DIR / "master_pct_normalized.csv")
v20_df = pd.read_csv(DATA_DIR / "task_statements_v20.1.csv", usecols=["Title"])
v30_df = pd.read_csv(DATA_DIR / "task_statements_v30.1.csv", usecols=["Title"])

# 2. Extract unique titles
master_titles = set(master_df['title'].dropna().unique())
v20_titles = set(v20_df['Title'].dropna().unique())
v30_titles = set(v30_df['Title'].dropna().unique())

# 3. Find discrepancies
# Titles in Master that are missing from the 2015 (v20.1) version
only_in_master = master_titles - v20_titles

print(f"Unique titles in Master: {len(master_titles)}")
print(f"Unique titles in O*NET v20.1 (2015): {len(v20_titles)}")
print(f"Titles in Master NOT in v20.1: {len(only_in_master)}")

# 4. Debug: Why are they missing?
# Check how many of these "extra" titles actually exist in the 2025 (v30.1) version
found_in_v30 = only_in_master.intersection(v30_titles)
still_missing = only_in_master - v30_titles

print(f"\nOut of those {len(only_in_master)} extra titles:")
print(f"  - Found in O*NET v30.1 (Newer jobs): {len(found_in_v30)}")
print(f"  - Truly unknown titles: {len(still_missing)}")

# 5. Check for normalization issues
# Sometimes they don't match because of "Nurse Practitioners" vs "Nurse Practitioner"
master_norm = {normalize_text(t) for t in still_missing}
v20_norm = {normalize_text(t) for t in v20_titles}
v30_norm = {normalize_text(t) for t in v30_titles}

true_discrepancies = [t for t in still_missing if normalize_text(t) not in v20_norm and normalize_text(t) not in v30_norm]

print(f"  - After normalization, truly unique discrepancies: {len(true_discrepancies)}")

# 6. Show some examples of the missing ones
if true_discrepancies:
    print("\nExample titles in Master that aren't in any O*NET version:")
    for t in sorted(true_discrepancies)[:15]:
        print(f"  - {t}")

# Save the discrepancies for manual review
if true_discrepancies:
    pd.Series(true_discrepancies).to_csv(DATA_DIR / "title_discrepancies_debug.csv", index=False)

Unique titles in Master: 1007
Unique titles in O*NET v20.1 (2015): 974
Titles in Master NOT in v20.1: 59

Out of those 59 extra titles:
  - Found in O*NET v30.1 (Newer jobs): 39
  - Truly unknown titles: 20
  - After normalization, truly unique discrepancies: 20

Example titles in Master that aren't in any O*NET version:
  - Agricultural and Food Science Technicians
  - Clinical, Counseling, and School Psychologists
  - Computer Occupations, All Other
  - Electrical and Electronic Engineering Technicians
  - Engineering Technicians, Except Drafters, All Other
  - Entertainers and Performers, Sports and Related Workers, All Other
  - First-Line Supervisors of Fire Fighting and Prevention Workers
  - First-Line Supervisors of Protective Service Workers, All Other
  - Geological and Petroleum Technicians
  - Health Technologists and Technicians, All Other
  - Managers, All Other
  - Mathematical Science Occupations, All Other
  - Media and Communication Equipment Workers, All Other
  - Pe

## GWA pct norm for Debiasing

### Copilot

In [4]:
import pandas as pd

# ── Load raw Microsoft IWA metrics ──
iwa_metrics = pd.read_csv("../data/iwa_metrics.csv")  # adjust path as needed
iwa_metrics = iwa_metrics[iwa_metrics[['share_user', 'share_ai']].max(axis=1) >= 0.0005].copy()
iwa_metrics['total_share'] = iwa_metrics['share_user'] + iwa_metrics['share_ai']

# ── Load task hierarchy to get GWA mapping ──
tasks_df = pd.read_csv("../data/tasks_dwa_iwa_gwa_v30.1_physical.csv")

# Normalize for merge
def normalize_text(s):
    return str(s).strip().lower()

iwa_metrics['norm_title'] = iwa_metrics['title'].apply(normalize_text)
tasks_df['norm_iwa_title'] = tasks_df['iwa_title'].apply(normalize_text)

# Get unique IWA → GWA mapping
iwa_to_gwa = tasks_df[['norm_iwa_title', 'gwa_title']].drop_duplicates('norm_iwa_title')

# Merge GWA onto IWA metrics
iwa_metrics = iwa_metrics.merge(iwa_to_gwa, left_on='norm_title', right_on='norm_iwa_title', how='left')

# Sum total_share by GWA, then normalize to %
gwa_shares = iwa_metrics.groupby('gwa_title')['total_share'].sum()
gwa_pct = (gwa_shares / gwa_shares.sum() * 100).sort_values(ascending=False)

print("Microsoft Copilot GWA Distribution (%):\n")
for gwa, pct in gwa_pct.items():
    print(f"  {pct:5.1f}%  {gwa}")
print(f"\n  Total: {gwa_pct.sum():.1f}%")

Microsoft Copilot GWA Distribution (%):

   24.3%  Getting Information
   15.4%  Communicating with People Outside the Organization
   12.7%  Performing for or Working Directly with the Public
    8.4%  Assisting and Caring for Others
    5.2%  Interpreting the Meaning of Information for Others
    5.1%  Documenting/Recording Information
    4.4%  Thinking Creatively
    3.5%  Providing Consultation and Advice to Others
    3.3%  Updating and Using Relevant Knowledge
    3.2%  Making Decisions and Solving Problems
    2.7%  Working with Computers
    2.2%  Communicating with Supervisors, Peers, or Subordinates
    1.4%  Analyzing Data or Information
    1.3%  Coaching and Developing Others
    1.3%  Training and Teaching Others
    1.0%  Judging the Qualities of Objects, Services, or People
    0.7%  Processing Information
    0.6%  Handling and Moving Objects
    0.6%  Selling or Influencing Others
    0.5%  Performing Administrative Activities
    0.5%  Monitoring Processes, Material

### ChatGPT

From the image (ChatGPT work-related GWA shares, Figure 15):

| GWA | Share (%) |
|-----|-----------|
| Documenting/Recording Information | 18.4 |
| Making Decisions and Solving Problems | 14.9 |
| Thinking Creatively | 13.0 |
| Working with Computers | 10.8 |
| Interpreting the Meaning of Information for Others | 10.1 |
| Getting Information | 9.3 |
| Providing Consultation and Advice to Others | 4.4 |
| Analyzing Data or Information | 3.0 |
| Communicating with Supervisors, Peers, or Subordinates | 2.8 |
| Judging the Qualities of Objects, Services, or People | 2.0 |
| Communicating with People Outside the Organization | 1.4 |
| Ambiguous | 1.1 |
| Estimating the Quantifiable Characteristics of Products, Events, or Information | 1.0 |
| Performing Administrative Activities | 1.0 |
| Training and Teaching Others | 0.8 |
| Selling or Influencing Others | 0.8 |
| Assisting and Caring for Others | 0.6 |
| Organizing, Planning, and Prioritizing Work | 0.6 |
| Scheduling Work and Activities | 0.5 |
| Developing Objectives and Strategies | 0.4 |
| Processing Information | 0.4 |
| Staffing Organizational Units | 0.3 |
| Updating and Using Relevant Knowledge | 0.3 |
| Resolving Conflicts and Negotiating with Others | 0.2 |
| Evaluating Information to Determine Compliance with Standards | 0.2 |
| Handling and Moving Objects | 0.2 |
| Coaching and Developing Others | 0.2 |
| Monitoring and Controlling Resources | 0.2 |
| Monitoring Processes, Materials, or Surroundings | 0.2 |
| Identifying Objects, Actions, and Events | 0.2 |
| Establishing and Maintaining Interpersonal Relationships | 0.2 |
| Inspecting Equipment, Structures, or Materials | 0.2 |
| Guiding, Directing, and Motivating Subordinates | 0.2 |
| Performing for or Working Directly with the Public | 0.1 |
| Repairing and Maintaining Mechanical Equipment | 0.1 |
| Performing General Physical Activities | 0.1 |
| Suppressed | 0.1 |

### Claude

In [5]:
import pandas as pd

aei = pd.read_csv("../data/final/final_aei_all_usage_2026-02-12.csv")

# Dedupe on unique chain
aei = aei.drop_duplicates(subset=['task_normalized', 'dwa_title', 'iwa_title', 'gwa_title'])

# Count how many chains each task maps to
aei['n_chains'] = aei.groupby('task_normalized')['gwa_title'].transform('count')

# Divide pct so each task's total contribution is preserved
aei['pct_adj'] = aei['pct_normalized'] / aei['n_chains']

# Sum by GWA and normalize to %
gwa_shares = aei.groupby('gwa_title')['pct_adj'].sum()
gwa_pct = (gwa_shares / gwa_shares.sum() * 100).sort_values(ascending=False)

print("AEI All Usage GWA Distribution (%):\n")
for gwa, pct in gwa_pct.items():
    print(f"  {pct:5.1f}%  {gwa}")
print(f"\n  Total: {gwa_pct.sum():.1f}%")

AEI All Usage GWA Distribution (%):

   33.7%  Thinking Creatively
   11.3%  Interacting With Computers
    8.7%  Documenting/Recording Information
    8.4%  Analyzing Data or Information
    4.0%  Provide Consultation and Advice to Others
    3.7%  Training and Teaching Others
    3.7%  Making Decisions and Solving Problems
    3.6%  Getting Information
    2.7%  Inspecting Equipment, Structures, or Material
    2.4%  Developing Objectives and Strategies
    2.2%  Judging the Qualities of Things, Services, or People
    2.0%  Interpreting the Meaning of Information for Others
    2.0%  Guiding, Directing, and Motivating Subordinates
    1.8%  Communicating with Supervisors, Peers, or Subordinates
    1.4%  Performing for or Working Directly with the Public
    1.3%  Processing Information
    0.9%  Communicating with Persons Outside Organization
    0.9%  Repairing and Maintaining Mechanical Equipment
    0.8%  Updating and Using Relevant Knowledge
    0.7%  Monitor Processes, Materia

C:\Users\teddy\AppData\Local\Temp\ipykernel_42024\1567501319.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aei['n_chains'] = aei.groupby('task_normalized')['gwa_title'].transform('count')
C:\Users\teddy\AppData\Local\Temp\ipykernel_42024\1567501319.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  aei['pct_adj'] = aei['pct_normalized'] / aei['n_chains']
